In [ ]:
!pip install -q transformers

import re
import os
import glob
import json
import unicodedata
from transformers import AutoTokenizer
from google.colab import files

In [ ]:
MODEL_NAME = "Wikidepia/IndoT5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Konfigurasi khusus arsitektur IndoT5 (maksimal encoder = 512 token)
ENCODER_MAX_TOKENS = 512
TASK_PREFIX = "ringkas: "
SAFETY_MARGIN_TOKENS = 16
OVERLAP_TOKENS = 50

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

prefix_tokens = count_tokens(TASK_PREFIX)
MAX_CHUNK_TOKENS = ENCODER_MAX_TOKENS - prefix_tokens - SAFETY_MARGIN_TOKENS

print(f"ALOKASI ANGGARAN TOKEN ({MODEL_NAME})")
print("="*60)
print(f"Kapasitas Maks Encoder  : {ENCODER_MAX_TOKENS} token")
print(f"(-) Task Prefix         : {prefix_tokens} token ('{TASK_PREFIX}')")
print(f"(-) Margin Keamanan     : {SAFETY_MARGIN_TOKENS} token")
print("-"*60)
print(f"ANGGARAN MAKS PER CHUNK : {MAX_CHUNK_TOKENS} token (Overlap: {OVERLAP_TOKENS} token)")

ALOKASI ANGGARAN TOKEN (Wikidepia/IndoT5-base)
Kapasitas Maks Encoder  : 512 token
(-) Task Prefix         : 2 token ('ringkas: ')
(-) Margin Keamanan     : 16 token
------------------------------------------------------------
ANGGARAN MAKS PER CHUNK : 494 token (Overlap: 50 token)


In [ ]:
print("Silakan upload file 'extracted_pages.json' (atau .txt hasil ekstraksi)...")
uploaded = files.upload()

def clean_page_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    # Sambungkan kata terpotong tanda hubung di akhir baris PDF
    text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)
    # Hapus deretan titik daftar isi
    text = re.sub(r"[.\-_]{4,}", " ", text)
    # Rapikan newline tunggal dalam satu paragraf menjadi spasi tanpa menghapus batas kalimat
    text = re.sub(r"\r\n|\r", "\n", text)
    paragraphs = re.split(r"\n\s*\n+", text)
    cleaned_p = [re.sub(r"\s+", " ", p).strip() for p in paragraphs if p.strip()]
    return " ".join(cleaned_p)

def split_into_sentences(text: str) -> list:
    protected = re.sub(
        r"\b(Dr|Prof|dll|dsb|dkk|hlm|No|Hlm|Fig|Tab|S\.Pd|M\.Kom|S\.Kom)\.",
        r"\1<PRD>",
        text
    )
    raw_sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"“(\[])', protected)
    return [s.replace("<PRD>", ".").strip() for s in raw_sentences if s.strip()]

if uploaded:
    input_filename = list(uploaded.keys())[-1]
    raw_bytes = uploaded[input_filename]
    pages_list = []

    if input_filename.lower().endswith(".json"):
        pages_list = json.loads(raw_bytes.decode("utf-8"))
    elif input_filename.lower().endswith(".txt"):
        raw_txt = raw_bytes.decode("utf-8", errors="ignore")
        parts = re.split(r"---\s*\[?Halaman\s*(\d+)\]?\s*---", raw_txt, flags=re.IGNORECASE)
        if len(parts) > 1:
            for i in range(1, len(parts), 2):
                pages_list.append({"page": int(parts[i]), "text": parts[i + 1]})
        else:
            pages_list.append({"page": 1, "text": raw_txt})

    global_idx = 1
    for p_item in pages_list:
        page_num = p_item["page"]
        cleaned = clean_page_text(p_item["text"])
        if not cleaned:
            continue

        for sent in split_into_sentences(cleaned):
            sentence_records.append({
                "sentence_id": f"S{global_idx:04d}",
                "page": page_num,
                "text": sent,
                "tokens": count_tokens(sent)
            })
            global_idx += 1

    if sentence_records:
        is_ready = True
        total_tok = sum(s["tokens"] for s in sentence_records)
        print(f"\n[BERHASIL] Terindeks {len(sentence_records)} kalimat dari {len(pages_list)} halaman ({total_tok:,} total token).")
    else:
        print("\n[DITOLAK] Tidak ditemukan kalimat valid di dalam file.")
else:
    print("\n[INFO] Upload dibatalkan. State sudah di-reset.")

Silakan upload file 'extracted_pages.json' (atau .txt hasil ekstraksi)...


Saving extracted_pages.json to extracted_pages.json

[BERHASIL] Terindeks 91 kalimat dari 5 halaman (1,833 total token).


In [ ]:
chunks = []

def split_oversized_sentence(sent_obj: dict, max_tokens: int) -> list:
    """Fallback jika ada 1 kalimat super panjang (misal tabel tanpa titik) yang melebihi anggaran."""
    token_ids = tokenizer.encode(sent_obj["text"], add_special_tokens=False)
    sub_parts = []
    for start in range(0, len(token_ids), max_tokens):
        sub_text = tokenizer.decode(token_ids[start:start + max_tokens], skip_special_tokens=True).strip()
        sub_parts.append({
            "sentence_id": sent_obj["sentence_id"],
            "page": sent_obj["page"],
            "text": sub_text,
            "tokens": len(token_ids[start:start + max_tokens])
        })
    return sub_parts

def build_token_aware_chunks(records: list, max_tokens: int, overlap_tokens: int) -> list:
    # Normalisasi jika ada kalimat tunggal melebihi max_tokens
    normalized_records = []
    for r in records:
        if r["tokens"] > max_tokens:
            normalized_records.extend(split_oversized_sentence(r, max_tokens))
        else:
            normalized_records.append(r)

    result = []
    current_sents = []

    def finalize_chunk(chunk_idx: int, sents: list) -> dict:
        joined_text = " ".join(s["text"] for s in sents)
        actual_tokens = count_tokens(joined_text)
        unique_pages = sorted(list({s["page"] for s in sents}))
        sent_ids = [s["sentence_id"] for s in sents]
        return {
            "chunk_id": f"CHUNK_{chunk_idx:03d}",
            "token_count": actual_tokens,
            "token_budget": max_tokens,
            "source_pages": unique_pages,
            "sentence_ids": sent_ids,
            "start_sentence_id": sent_ids[0],
            "end_sentence_id": sent_ids[-1],
            "text": joined_text,
            "sentences_detail": [
                {"sentence_id": s["sentence_id"], "page": s["page"], "text": s["text"]}
                for s in sents
            ]
        }

    for sent in normalized_records:
        candidate_sents = current_sents + [sent]
        candidate_text = " ".join(s["text"] for s in candidate_sents)
        candidate_tokens = count_tokens(candidate_text)

        if candidate_tokens > max_tokens and current_sents:
            result.append(finalize_chunk(len(result) + 1, current_sents))

            # Ambil kalimat terakhir sebagai overlap (maksimal sesuai overlap_tokens)
            overlap_sents = []
            for prev_s in reversed(current_sents):
                test_overlap = [prev_s] + overlap_sents
                if count_tokens(" ".join(s["text"] for s in test_overlap)) <= overlap_tokens:
                    overlap_sents = test_overlap
                else:
                    break

            current_sents = overlap_sents + [sent]
        else:
            current_sents = candidate_sents

    if current_sents:
        result.append(finalize_chunk(len(result) + 1, current_sents))

    return result

if not is_ready:
    print("[STOP] Jalankan Cell 3 dan pastikan materi sudah terindeks.")
else:
    chunks = build_token_aware_chunks(sentence_records, MAX_CHUNK_TOKENS, OVERLAP_TOKENS)
    print(f"[SUKSES] Terbentuk {len(chunks)} chunk yang 100% muat dalam anggaran {MAX_CHUNK_TOKENS:,} token.")

[SUKSES] Terbentuk 4 chunk yang 100% muat dalam anggaran 494 token.


In [ ]:
if not chunks:
    print("[STOP] Belum ada chunk yang terbentuk. Jalankan Cell 4 terlebih dahulu.")
else:
    print("="*75)
    print("AUDIT CHUNK & PEMETAAN TRACEABILITY")
    print("="*75)

    for c in chunks:
        status = "OK" if c["token_count"] <= MAX_CHUNK_TOKENS else "OVERFLOW"
        pages_str = ",".join(map(str, c["source_pages"]))
        print(
            f"{c['chunk_id']} | {c['token_count']:4d}/{MAX_CHUNK_TOKENS} tok [{status}] | "
            f"Hal: [{pages_str}] | Kalimat: {c['start_sentence_id']}..{c['end_sentence_id']} ({len(c['sentence_ids'])} kal)"
        )

    output_payload = {
        "pipeline_metadata": {
            "source_file": input_filename,
            "tokenizer_model": MODEL_NAME,
            "max_chunk_tokens": MAX_CHUNK_TOKENS,
            "overlap_tokens": OVERLAP_TOKENS,
            "total_chunks": len(chunks),
            "total_sentences": len(sentence_records)
        },
        "chunks": chunks
    }

    base_name = input_filename.rsplit(".", 1)[0] if input_filename else "materi_utbk"
    output_file = f"{base_name}_token_chunks.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(output_payload, f, ensure_ascii=False, indent=2)

    print("="*75)
    print(f"File output siap digunakan untuk tahap generasi UTBK: {output_file}")
    files.download(output_file)

AUDIT CHUNK & PEMETAAN TRACEABILITY
CHUNK_001 |  485/494 tok [OK] | Hal: [1,2,3] | Kalimat: S0001..S0018 (18 kal)
CHUNK_002 |  494/494 tok [OK] | Hal: [3,4] | Kalimat: S0018..S0047 (30 kal)
CHUNK_003 |  488/494 tok [OK] | Hal: [4,5] | Kalimat: S0046..S0070 (25 kal)
CHUNK_004 |  428/494 tok [OK] | Hal: [5] | Kalimat: S0070..S0091 (22 kal)
File output siap digunakan untuk tahap generasi UTBK: extracted_pages_token_chunks.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>